# Optimization Tricks in Deep Learning

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 04.07 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/08_optimization_in_deep_learning.ipynb)

---

## 🎯 Learning Objective

Learn the practical optimization tricks — gradient clipping, gradient accumulation, mixed precision — that make training large models feasible.

---

## 📐 Theory

Everything so far in this module has been about *which direction and how far* to step. This
notebook covers three practical realities that determine whether a training run survives long
enough to converge at all, once you leave toy problems for real hardware and real data.

### Gradient clipping

A single unusual batch — an outlier, a numerically unstable loss spike, an unlucky sample near
a discontinuity — can produce a gradient with enormous norm. Combined with a learning rate
tuned for *typical* gradients, one such batch can push weights far outside any region where the
loss behaves sensibly, and training may never recover (recall the exploding-gradient mechanism
from `02.10`, now triggered by data rather than depth). **Gradient clipping by norm** caps the
*overall* gradient vector's length before the update:

$$\mathbf{g} \leftarrow \mathbf{g} \cdot \min\!\left(1, \frac{\tau}{\|\mathbf{g}\|}\right)$$

for a threshold $\tau$. If $\|\mathbf{g}\|\le\tau$ already, nothing changes; if
$\|\mathbf{g}\|>\tau$, the vector is rescaled down to exactly length $\tau$ while keeping its
*direction* — clipping doesn't change which way you step, only how far, which matters because
the direction is still a locally valid descent direction even when its magnitude is untrustworthy.

### Gradient accumulation

GPU memory limits how large a batch you can fit at once. **Gradient accumulation** simulates a
larger effective batch by running several smaller "micro-batches," summing (or averaging) their
gradients *without* resetting between them, and only calling the optimizer step once every $k$
micro-batches:

$$\mathbf{g}_{\text{effective}} = \frac{1}{k}\sum_{i=1}^{k} \mathbf{g}_i$$

If each micro-batch loss is itself an average over its own examples, averaging the $k$
micro-batch gradients reproduces *exactly* the gradient you would have gotten from one large
batch containing all $k$ micro-batches' examples together (provided the micro-batches are equal
size) — accumulation trades memory for a few extra forward/backward passes, not for accuracy.

### Mixed precision training

Modern GPUs compute much faster in float16/bfloat16 than float32 (previewed in `06.01`, fully
covered in `06.06`), but naive float16 training risks numerical underflow: gradients that would
be representable in float32 can round to exactly zero in float16's narrower range. **Mixed
precision training** keeps a float32 "master copy" of the weights, computes the forward/backward
pass in float16 for speed, and (in the classic recipe) *scales the loss up* by a large constant
before backpropagating so small gradients don't underflow, then unscales the resulting gradients
before the optimizer step — recovering the precision-sensitive parts of training while keeping
the fast, cheap arithmetic everywhere else.

### Why these are "engineering," not new math

None of this changes *what* is being optimized — clipping still points along $-\nabla L$
(direction unchanged, magnitude capped), accumulation still computes the exact mini-batch
gradient (just via smaller pieces), mixed precision still computes the same forward/backward
formulas (just in a different number format with compensating scale factors). They're solutions
to *engineering* constraints — unstable batches, limited memory, limited numeric range — layered
on top of the pure mathematics from `04.01`-`04.07`, which is exactly why they combine so
easily with any optimizer covered so far.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A single corrupted training batch spikes the loss for an unclipped run and takes many steps to
recover from; the same run with gradient clipping barely notices the same corrupted batch.

In [ ]:
rng = np.random.default_rng(0)
torch.manual_seed(0)
N = 300
X = torch.tensor(rng.uniform(-2, 2, size=(N, 5)), dtype=torch.float32)
true_w = torch.tensor(rng.normal(size=5), dtype=torch.float32)
y = (X @ true_w + 0.1 * torch.randn(N)).unsqueeze(1)

def make_model():
    torch.manual_seed(1)
    return torch.nn.Sequential(torch.nn.Linear(5, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))

def train(clip_norm, steps=150, batch_size=10, lr=0.1, seed=2, outlier_step=30):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    losses = []
    for t in range(steps):
        idx = torch.randint(0, N, (batch_size,), generator=g)
        xb, yb = X[idx].clone(), y[idx].clone()
        if t == outlier_step:
            yb = yb + 200.0  # simulate one corrupted/outlier batch, a single transient event
        opt.zero_grad()
        loss = torch.nn.functional.mse_loss(model(xb), yb)
        loss.backward()
        if clip_norm:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        opt.step()
        with torch.no_grad():
            losses.append(torch.nn.functional.mse_loss(model(X), y).item())
    return losses

losses_noclip = train(clip_norm=None)
losses_clip = train(clip_norm=1.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy([min(l, 1e5) for l in losses_noclip], color="#C44E52", label="no clipping")
ax.semilogy([min(l, 1e5) for l in losses_clip], color="#4C72B0", label="clip_grad_norm=1.0")
ax.axvline(30, color="gray", ls="--", lw=1, label="corrupted batch hits here")
ax.set_xlabel("training step"); ax.set_ylabel("full-dataset loss (log scale)")
ax.set_title("One bad batch derails unclipped training; clipping absorbs it")
ax.legend()
plt.show()

print(f"Peak loss after the outlier -- no clip: {max(losses_noclip):.2f}, clipped: {max(losses_clip):.2f}")
print(f"Final loss -- no clip: {losses_noclip[-1]:.4f}, clipped: {losses_clip[-1]:.4f}")

## 🐍 Implementation from Scratch

We implement gradient-norm clipping by hand and confirm it matches `torch.nn.utils.clip_grad_norm_`
exactly, then verify that gradient accumulation over several micro-batches reproduces the exact
gradient of one large batch.

In [ ]:
def clip_grad_norm_scratch(parameters, max_norm):
    # model.parameters() returns a GENERATOR -- materialize it to a list first, or the norm
    # computation below would exhaust it, leaving nothing for the mutation loop afterward.
    parameters = list(parameters)
    # Compute the TOTAL norm across all parameters jointly (not per-parameter), matching
    # torch's convention: treat every gradient as one big concatenated vector.
    total_norm_sq = sum((p.grad ** 2).sum().item() for p in parameters if p.grad is not None)
    total_norm = total_norm_sq ** 0.5
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1:
        for p in parameters:
            if p.grad is not None:
                p.grad.mul_(clip_coef)
    return total_norm

# Verify against torch's built-in on a model with a deliberately huge gradient
torch.manual_seed(3)
model_a = torch.nn.Linear(4, 1)
model_b = torch.nn.Linear(4, 1)
model_b.load_state_dict(model_a.state_dict())  # identical starting weights

x_huge = torch.randn(4) * 100  # large input -> large gradient
for model in (model_a, model_b):
    model.zero_grad()
    loss = (model(x_huge) ** 2).sum()
    loss.backward()

norm_scratch = clip_grad_norm_scratch(model_a.parameters(), max_norm=1.0)
norm_torch = torch.nn.utils.clip_grad_norm_(model_b.parameters(), max_norm=1.0)
print(f"Pre-clip gradient norm: {norm_scratch:.4f} (scratch) vs {norm_torch.item():.4f} (torch)")
grads_match = all(torch.allclose(pa.grad, pb.grad, atol=1e-5) for pa, pb in zip(model_a.parameters(), model_b.parameters()))
print(f"Post-clip gradients match exactly: {grads_match}")

# --- Gradient accumulation: k micro-batches should reproduce one big batch's gradient exactly ---
rng2 = np.random.default_rng(5)
N2 = 64
X2 = torch.tensor(rng2.uniform(-2, 2, size=(N2, 5)), dtype=torch.float32)
w2 = torch.tensor(rng2.normal(size=5), dtype=torch.float32)
y2 = (X2 @ w2 + 0.1 * torch.randn(N2)).unsqueeze(1)

torch.manual_seed(1)
model_big = torch.nn.Sequential(torch.nn.Linear(5, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
torch.manual_seed(1)
model_accum = torch.nn.Sequential(torch.nn.Linear(5, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))

model_big.zero_grad()
torch.nn.functional.mse_loss(model_big(X2), y2).backward()

model_accum.zero_grad()
micro_bs, n_micro = 16, N2 // 16
for i in range(n_micro):
    xb, yb = X2[i*micro_bs:(i+1)*micro_bs], y2[i*micro_bs:(i+1)*micro_bs]
    # Divide by n_micro so summed-then-accumulated gradients average correctly across
    # equal-sized micro-batches -- this is the "loss / accumulation_steps" convention
    (torch.nn.functional.mse_loss(model_accum(xb), yb) / n_micro).backward()

max_diff = max((pb.grad - pa.grad).abs().max().item() for pa, pb in zip(model_big.parameters(), model_accum.parameters()))
print(f"\nOne big batch (size {N2}) vs {n_micro} accumulated micro-batches (size {micro_bs} each):")
print(f"max gradient difference across all parameters: {max_diff:.2e} (floating-point-level agreement)")

## 🤖 Why This Matters for AI

Every large language model's public training recipe mentions gradient clipping (almost always
by global norm, typically $\tau=1.0$) — it's cheap insurance against the rare batch that would
otherwise derail days of training. Gradient accumulation is how models with effective batch
sizes in the millions of tokens get trained on GPUs that can only fit a few thousand tokens at
once: run many small forward/backward passes, accumulate, then step. Mixed precision (bfloat16
compute, float32 master weights) is standard for training anything at LLM scale, since it can
roughly double throughput on modern accelerators. Below, we combine all three ideas into one
small training loop — clipped, accumulated, and run in float16 — and confirm the combination
still trains correctly, matching a full-precision, non-accumulated baseline.

In [ ]:
# 1) Float16 underflow, and the loss-scaling trick that mixed-precision training relies on
tiny_grad = torch.tensor(1e-8, dtype=torch.float32)  # a small but perfectly valid float32 gradient
print(f"Gradient value in float32: {tiny_grad.item():.3e}")
print(f"Same value cast directly to float16: {tiny_grad.half().item():.3e}  <- UNDERFLOWED TO ZERO")

scale = 4096.0
scaled_then_cast = (tiny_grad * scale).half()
recovered = (scaled_then_cast.float() / scale).item()
print(f"\nScale by {scale} BEFORE casting to float16: {scaled_then_cast.item():.3e}")
print(f"Unscale by dividing by {scale} AFTER: {recovered:.3e}")
print(f"Relative error vs. the true float32 value: {abs(recovered - tiny_grad.item())/tiny_grad.item()*100:.4f}%")
print("This scale-before/unscale-after pattern is EXACTLY what GradScaler does automatically.")

# 2) Combine clipping + accumulation + autocast (bfloat16 on CPU) into one realistic training loop
def train_mixed_precision(steps=100, accum_steps=4, micro_bs=8, clip_norm=1.0, lr=0.1):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(7)
    losses = []
    for step in range(steps):
        opt.zero_grad()
        for _ in range(accum_steps):
            idx = torch.randint(0, N, (micro_bs,), generator=g)
            with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
                pred = model(X[idx])
                loss = torch.nn.functional.mse_loss(pred, y[idx]) / accum_steps
            loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        opt.step()
        with torch.no_grad():
            losses.append(torch.nn.functional.mse_loss(model(X), y).item())
    return losses

losses_combo = train_mixed_precision()
print(f"\nClipped + accumulated + bfloat16-autocast training: loss {losses_combo[0]:.4f} -> {losses_combo[-1]:.4f}")
print("All three engineering tricks compose cleanly -- none of them changed WHAT is being optimized.")

## 🏋️ Exercises

### Warm-up
1. A model has two parameters with gradients $g_1=(3,4)$ and $g_2=(0,12)$ (as flat vectors).
   Compute the total gradient norm by hand, then compute the clipped gradients for
   `max_norm=5`. Verify with `clip_grad_norm_scratch`.

### Practice
2. In the Visual Intuition cell, change `outlier_step` to occur at step 100 instead of 30, and
   re-run both the clipped and unclipped versions. Does clipping still help as much? Explain
   your observation in terms of how much "progress" each run has already made before the
   outlier hits.

### Challenge
3. Extend the gradient accumulation check to use *unequal* micro-batch sizes (e.g. sizes
   20, 20, 24 instead of three equal batches of size ~21). Show that naively dividing by
   `n_micro` (the number of micro-batches) no longer exactly reproduces the single-large-batch
   gradient, and derive the correct weighted-averaging fix.

---

## 📚 Further Reading
- Pascanu, Mikolov & Bengio, ["On the difficulty of training Recurrent Neural Networks"](https://arxiv.org/abs/1211.5063) (gradient clipping)
- Micikevicius et al., ["Mixed Precision Training"](https://arxiv.org/abs/1710.03740)

---

*Next notebook: [Visualizing Loss Landscapes](09_loss_landscape_visualization.ipynb)*